# 🗣️ VoiceWeaver — 雲端算力（Colab GPU）

把 **語音合成（GPT-SoVITS）** 的算力搬到 Colab 免費 GPU；可自由加開 **生圖（SD-Turbo）** 與 **文字／復健評分（Qwen）**。手機 App 與網頁版都能連上，由雲端代跑。

> **為什麼不用 Google Apps Script？** 它是 Workspace 的 JavaScript 自動化，**沒有 GPU、不能跑 Python/PyTorch**，無法執行這些模型。GPU 推論只能用 Colab。

### 運作方式
```
  [手機 App]─┐                                   ┌ api_v2 :9880  語音 GPT-SoVITS
           ├ https 公開網址(Cloudflare) ─ 橋接:9879 ┼ :9881         生圖 SD-Turbo（選用）
  [網頁版] ─┘   app 走 /tts /models；web 走 /speak /voices /image /text（含 CORS）  └ :9882  文字 Qwen（選用）
```

### 事前準備（一次）
1. **語音模型上傳 Google Drive**：`我的雲端硬碟/VoiceWeaver/voice_models/`（＝手機 `/sdcard/VoiceWeaver/voice_models/` 那整包）。只跑生圖/文字、不跑語音則可略過。
2. Colab 上方 **執行階段 → 變更執行階段類型 → T4 GPU**。

> ⚠️ 不含任何語音模型；由你自行上傳有權使用的素材。GPT-SoVITS 為 AGPL-3.0。


## ① 自由選擇：這台 Colab 要跑哪些功能


In [ ]:
# 想跑哪個就設 True（語音最吃準備、生圖/文字可獨立開）。T4 記憶體有限，三個全開可能吃緊。
RUN_VOICE = True    # GPT-SoVITS 語音合成（:9880）
RUN_IMAGE = False   # SD-Turbo 生圖（:9881）
RUN_TEXT  = False   # Qwen 文字推論／復健評分（:9882）
assert RUN_VOICE or RUN_IMAGE or RUN_TEXT, '至少要開一個功能'
print('將啟動：', ' '.join(k for k,v in [('語音',RUN_VOICE),('生圖',RUN_IMAGE),('文字',RUN_TEXT)] if v))


## ② 確認 GPU


In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout.split(chr(10))[8] if r.returncode==0 and len(r.stdout.split(chr(10)))>8 else r.stdout or r.stderr)
assert r.returncode==0, '❌ 沒有 GPU！執行階段 → 變更執行階段類型 → GPU (T4)，再重跑。'
print('✅ GPU 就緒')


## ③ 安裝相依（約 3–6 分鐘）


In [ ]:
import os
ROOT='/content/GPT-SoVITS'
if RUN_VOICE and not os.path.exists(ROOT):
    !git clone --depth 1 https://github.com/RVC-Boss/GPT-SoVITS $ROOT
    !cd $ROOT && pip -q install -r requirements.txt 2>&1 | tail -3
if RUN_IMAGE:
    !pip -q install diffusers transformers accelerate 2>&1 | tail -2
if RUN_TEXT:
    !pip -q install 'transformers>=4.40' accelerate 2>&1 | tail -2
!pip -q install 'huggingface_hub>=0.23'
print('✅ 安裝完成')


## ④ 語音模型（RUN_VOICE 才需要）— 掛 Drive、下載底模、修正 ref 路徑


In [ ]:
import os, json, glob, shutil
if RUN_VOICE:
    from huggingface_hub import snapshot_download
    snapshot_download('lj1995/GPT-SoVITS', local_dir=f'{ROOT}/GPT_SoVITS/pretrained_models', local_dir_use_symlinks=False)
    from google.colab import drive; drive.mount('/content/drive')
    SRC='/content/drive/MyDrive/VoiceWeaver/voice_models'
    assert os.path.isdir(SRC), f'找不到 {SRC}（把手機 voice_models 上傳到 Drive 的 VoiceWeaver/ 底下）'
    for sub in os.listdir(SRC):
        s=os.path.join(SRC,sub); d=os.path.join(ROOT,sub)
        if os.path.isdir(s):
            for item in glob.glob(os.path.join(s,'**'),recursive=True):
                rel=os.path.relpath(item,s); dst=os.path.join(d,rel)
                if os.path.isdir(item): os.makedirs(dst,exist_ok=True)
                elif not os.path.exists(dst): os.makedirs(os.path.dirname(dst),exist_ok=True); shutil.copy2(item,dst)
    recs=json.load(open(os.path.join(SRC,'user_models','model_history.json'),encoding='utf-8'))
    def _ref(p):
        b=os.path.basename(p or '')
        for rt,_,fs in os.walk(os.path.join(ROOT,'ref_audio')):
            if b in fs: return os.path.join(rt,b)
        return p
    for r in recs: r['ref_path']=_ref(r.get('ref_path',''))
    os.makedirs(f'{ROOT}/user_models',exist_ok=True)
    json.dump(recs,open(f'{ROOT}/user_models/model_history.json','w',encoding='utf-8'),ensure_ascii=False,indent=2)
    print(f'✅ 載入 {len(recs)} 個語音模型：', ', '.join(r.get('name','') for r in recs))
else:
    ROOT='/content'; os.makedirs(f'{ROOT}/user_models',exist_ok=True)
    open(f'{ROOT}/user_models/model_history.json','w').write('[]')
    print('（跳過語音）')


## ⑤ 啟動所選服務


In [ ]:
import base64, subprocess, os, time, urllib.request
def _wait(url,label,secs=150):
    for _ in range(secs//3):
        time.sleep(3)
        try:
            with urllib.request.urlopen(url,timeout=3) as r:
                if r.status==200: print(f'✅ {label} 就緒'); return True
        except Exception: pass
    print(f'⚠️ {label} 逾時'); return False
procs={}
if RUN_VOICE:
    os.chdir(ROOT)
    procs['voice']=subprocess.Popen(['python','api_v2.py','-a','127.0.0.1','-p','9880'],
        stdout=open('/content/voice.log','w'),stderr=subprocess.STDOUT)
    _wait('http://127.0.0.1:9880/health','語音 api_v2')
if RUN_IMAGE:
    open('/content/img.py','wb').write(base64.b64decode('IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJWb2ljZVdlYXZlciDmnKzmqZ/nlJ/lnJbkvLrmnI3lmajvvIhNYWMg56uv77yM5YWN6YeR6ZGw77yJ44CCCgrot58gR1BULVNvVklUUyDkuIDmqKPlnKjjgIzpm7vohabmnKzmqZ/jgI3pgYvnrpfvvJrmiYvmqZ/pgI/pgY4gVVNCKGFkYiByZXZlcnNlKSDmiJYgV2ktRmkg6YCj5YiwIDo5ODgx77yMCuS4jemcgOimgeS7u+S9lembsuerryBBUEkg6YeR6ZGw44CC55SoIGRpZmZ1c2VycyArIEFwcGxlIE1QUyDot5EgU0QtVHVyYm/vvIjlv6vjgIE1MTJweOOAgemBqeWQiCBBQUMg5ZyW5Y2h77yJ44CCCgrnq6/pu57vvJoKICAgIEdFVCAgL2hlYWx0aCAgICAgICAgICAgICAtPiB7b2ssIHJlYWR5LCBtb2RlbCwgZGV2aWNlLCBlcnJvcn0KICAgIFBPU1QgL2dlbmVyYXRlICAgICAgICAgICBib2R5IHsicHJvbXB0IjogIi4uLiIsICJzdGVwcyI6IDJ9ICAtPiBpbWFnZS9wbmcKCuaooeWei+WPr+eUqOeSsOWig+iuiuaVuOimhuiTi++8mgogICAgSU1BR0VfR0VOX01PREVM77yI6aCQ6KitIHN0YWJpbGl0eWFpL3NkLXR1cmJv77yb5oOz5pu057K+57e75Y+v55SoIHN0YWJpbGl0eWFpL3NkeGwtdHVyYm/vvIkKICAgIElNQUdFX0dFTl9TVEVQU++8iOmgkOiorSAy77yJCiAgICBWT0lDRVdFQVZFUl9JTUFHRV9QT1JU77yI6aCQ6KitIDk4ODHvvIkKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBpbwppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHJlCmltcG9ydCBzb2NrZXQKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGhyZWFkaW5nCmltcG9ydCB0aW1lCmZyb20gaHR0cC5zZXJ2ZXIgaW1wb3J0IEJhc2VIVFRQUmVxdWVzdEhhbmRsZXIsIFRocmVhZGluZ0hUVFBTZXJ2ZXIKCgpkZWYgc2VsZl9pbmZvKCkgLT4gZGljdDoKICAgICIiIuWbnuWgseacrOapn+S4u+apn+WQjeiIhyBUYWlsc2NhbGUgSVDvvIgxMDAuNjQuMC4wLzEw77yJ77yM57Wm5omL5qmf6Ieq5YuV5Yqg5YWl6Zu76IWm5riF5Zau55So44CCIiIiCiAgICBob3N0ID0gc29ja2V0LmdldGhvc3RuYW1lKCkuc3BsaXQoIi4iKVswXSBvciAi6Zu76IWmIgogICAgdHNpcCA9ICIiCiAgICB0cnk6CiAgICAgICAgb3V0ID0gc3VicHJvY2Vzcy5ydW4oWyJpZmNvbmZpZyJdLCBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUsIHRpbWVvdXQ9NCkuc3Rkb3V0CiAgICAgICAgZm9yIG0gaW4gcmUuZmluZGl0ZXIociJpbmV0ICgxMDBcLlxkK1wuXGQrXC5cZCspIiwgb3V0KToKICAgICAgICAgICAgaXAgPSBtLmdyb3VwKDEpCiAgICAgICAgICAgIGlmIDY0IDw9IGludChpcC5zcGxpdCgiLiIpWzFdKSA8PSAxMjc6ICAgIyAxMDAuNjQuMC4wLzEwCiAgICAgICAgICAgICAgICB0c2lwID0gaXAKICAgICAgICAgICAgICAgIGJyZWFrCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHBhc3MKICAgIHJldHVybiB7Im9rIjogVHJ1ZSwgImhvc3RuYW1lIjogaG9zdCwgInRhaWxzY2FsZV9pcCI6IHRzaXB9CgpQT1JUID0gaW50KG9zLmVudmlyb24uZ2V0KCJWT0lDRVdFQVZFUl9JTUFHRV9QT1JUIiwgIjk4ODEiKSkKTU9ERUwgPSBvcy5lbnZpcm9uLmdldCgiSU1BR0VfR0VOX01PREVMIiwgInN0YWJpbGl0eWFpL3NkLXR1cmJvIikKREVGQVVMVF9TVEVQUyA9IGludChvcy5lbnZpcm9uLmdldCgiSU1BR0VfR0VOX1NURVBTIiwgIjIiKSkKCl9waXBlID0gTm9uZQpfbG9jayA9IHRocmVhZGluZy5Mb2NrKCkgICAgICAgIyDllq7lvLXluo/liJfljJbvvIzpgb/lhY0gTVBTIOiomOaGtumrlOeIhuaOiQpfbG9hZF9lcnIgPSAiIgpfZGV2aWNlID0gImNwdSIKX2J1c3kgPSAwICAgICAgICAgICAgICAgICAgICAgICMg6YCy6KGM5Lit55qE55Sf5ZyW5pW477yI57Wm5o6n5Yi25Lit5b+D6aGv56S644CM5ZCI5oiQ6YCy5bqm44CN77yJCl9kb25lID0gMCAgICAgICAgICAgICAgICAgICAgICMg57Sv6KiI5a6M5oiQ5by15pW4Cl9sYXN0X3Byb21wdCA9ICIiCgoKZGVmIGxvYWRfcGlwZSgpIC0+IE5vbmU6CiAgICAiIiLog4zmma/ovInlhaXmqKHlnovvvIjpppbmrKHmnIPoh6rli5XkuIvovInvvInvvIzorpMgc2VydmVyIOiDveeri+WIu+WbnuaHiSAvaGVhbHRo44CCIiIiCiAgICBnbG9iYWwgX3BpcGUsIF9sb2FkX2VyciwgX2RldmljZQogICAgdHJ5OgogICAgICAgIGltcG9ydCB0b3JjaAogICAgICAgIGZyb20gZGlmZnVzZXJzIGltcG9ydCBBdXRvUGlwZWxpbmVGb3JUZXh0MkltYWdlCgogICAgICAgICMg6KOd572u5YSq5YWI5bqP77yaQ1VEQe+8iFdpbmRvd3MvTGludXggTlZJRElB77yJ4oaSIE1QU++8iEFwcGxl77yJ4oaSIENQVeOAggogICAgICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgICAgIF9kZXZpY2UgPSAiY3VkYSIKICAgICAgICBlbGlmIGdldGF0dHIodG9yY2guYmFja2VuZHMsICJtcHMiLCBOb25lKSBpcyBub3QgTm9uZSBhbmQgdG9yY2guYmFja2VuZHMubXBzLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICBfZGV2aWNlID0gIm1wcyIKICAgICAgICBlbHNlOgogICAgICAgICAgICBfZGV2aWNlID0gImNwdSIKICAgICAgICBkdHlwZSA9IHRvcmNoLmZsb2F0MTYgaWYgX2RldmljZSBpbiAoImN1ZGEiLCAibXBzIikgZWxzZSB0b3JjaC5mbG9hdDMyCiAgICAgICAgIyDlhKrlhYjmipMgZnAxNiDororpq5TvvIjpq5TnqY3muJvljYrjgIFHUFUg6YGp55So77yJ77yb5rKS5pyJIGZwMTYg6K6K6auU55qE5qih5Z6L5YaN6YCA5Zue6aCQ6Kit5qyK6YeNCiAgICAgICAgdHJ5OgogICAgICAgICAgICBwaXBlID0gQXV0b1BpcGVsaW5lRm9yVGV4dDJJbWFnZS5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgICAgICAgICBNT0RFTCwgdG9yY2hfZHR5cGU9ZHR5cGUsIHZhcmlhbnQ9ImZwMTYiLCB1c2Vfc2FmZXRlbnNvcnM9VHJ1ZQogICAgICAgICAgICApCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgcGlwZSA9IEF1dG9QaXBlbGluZUZvclRleHQySW1hZ2UuZnJvbV9wcmV0cmFpbmVkKE1PREVMLCB0b3JjaF9kdHlwZT1kdHlwZSkKICAgICAgICBwaXBlID0gcGlwZS50byhfZGV2aWNlKQogICAgICAgIHBpcGUuc2V0X3Byb2dyZXNzX2Jhcl9jb25maWcoZGlzYWJsZT1UcnVlKQogICAgICAgICMgQUFDIOeUqOmAlO+8mumXnOaOiSBzYWZldHkgY2hlY2tlcu+8jOmBv+WFjeiqpOWIpOaKiuWcluWhl+m7kQogICAgICAgIGlmIGhhc2F0dHIocGlwZSwgInNhZmV0eV9jaGVja2VyIik6CiAgICAgICAgICAgIHBpcGUuc2FmZXR5X2NoZWNrZXIgPSBOb25lCiAgICAgICAgX3BpcGUgPSBwaXBlCiAgICAgICAgcHJpbnQoZiJbaW1hZ2VdIG1vZGVsIHJlYWR5OiB7TU9ERUx9IG9uIHtfZGV2aWNlfSIsIGZsdXNoPVRydWUpCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzogICMgbm9xYTogQkxFMDAxCiAgICAgICAgX2xvYWRfZXJyID0gc3RyKGV4YykKICAgICAgICBwcmludChmIltpbWFnZV0gbG9hZCBmYWlsZWQ6IHtleGN9IiwgZmx1c2g9VHJ1ZSkKCgpkZWYgZ2VuZXJhdGVfcG5nKHByb21wdDogc3RyLCBzdGVwczogaW50KSAtPiBieXRlczoKICAgIGdsb2JhbCBfYnVzeSwgX2RvbmUsIF9sYXN0X3Byb21wdAogICAgX2J1c3kgKz0gMQogICAgX2xhc3RfcHJvbXB0ID0gcHJvbXB0CiAgICBwcmludChmIltpbWFnZV0gZ2VuZXJhdGluZyAoe19idXN5fSBpbiBxdWV1ZSk6IHtwcm9tcHRbOjYwXX0iLCBmbHVzaD1UcnVlKQogICAgdHJ5OgogICAgICAgIHdpdGggX2xvY2s6CiAgICAgICAgICAgIGltYWdlID0gX3BpcGUoCiAgICAgICAgICAgICAgICBwcm9tcHQ9cHJvbXB0LAogICAgICAgICAgICAgICAgbnVtX2luZmVyZW5jZV9zdGVwcz1tYXgoMSwgbWluKHN0ZXBzLCA4KSksCiAgICAgICAgICAgICAgICBndWlkYW5jZV9zY2FsZT0wLjAsICAgIyB0dXJibyDns7vliJflm7rlrprnlKggMAogICAgICAgICAgICApLmltYWdlc1swXQogICAgICAgIGJ1ZiA9IGlvLkJ5dGVzSU8oKQogICAgICAgIGltYWdlLnNhdmUoYnVmLCBmb3JtYXQ9IlBORyIpCiAgICAgICAgX2RvbmUgKz0gMQogICAgICAgIHByaW50KGYiW2ltYWdlXSBkb25lICh0b3RhbCB7X2RvbmV9KToge3Byb21wdFs6NDBdfSIsIGZsdXNoPVRydWUpCiAgICAgICAgcmV0dXJuIGJ1Zi5nZXR2YWx1ZSgpCiAgICBmaW5hbGx5OgogICAgICAgIF9idXN5IC09IDEKCgpjbGFzcyBIYW5kbGVyKEJhc2VIVFRQUmVxdWVzdEhhbmRsZXIpOgogICAgZGVmIF9qc29uKHNlbGYsIGNvZGU6IGludCwgb2JqOiBkaWN0KSAtPiBOb25lOgogICAgICAgIGIgPSBqc29uLmR1bXBzKG9iaiwgZW5zdXJlX2FzY2lpPUZhbHNlKS5lbmNvZGUoInV0Zi04IikKICAgICAgICBzZWxmLnNlbmRfcmVzcG9uc2UoY29kZSkKICAgICAgICBzZWxmLnNlbmRfaGVhZGVyKCJDb250ZW50LVR5cGUiLCAiYXBwbGljYXRpb24vanNvbjsgY2hhcnNldD11dGYtOCIpCiAgICAgICAgc2VsZi5zZW5kX2hlYWRlcigiQ29udGVudC1MZW5ndGgiLCBzdHIobGVuKGIpKSkKICAgICAgICBzZWxmLmVuZF9oZWFkZXJzKCkKICAgICAgICBzZWxmLndmaWxlLndyaXRlKGIpCgogICAgZGVmIGRvX0dFVChzZWxmKSAtPiBOb25lOgogICAgICAgIGlmIHNlbGYucGF0aC5zdGFydHN3aXRoKCIvd2hvYW1pIik6CiAgICAgICAgICAgIHNlbGYuX2pzb24oMjAwLCBzZWxmX2luZm8oKSkKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgaWYgc2VsZi5wYXRoLnN0YXJ0c3dpdGgoIi9iZW5jaCIpOgogICAgICAgICAgICAjIOWQnuWQkOmHj+mHj+a4rO+8muWbnuWbuuWumuWkp+Wwj+S9jeWFg+e1hOiuk+aJi+apn+err+ioiOaZguS8sOeul+mAo+e3mumAn+W6pu+8iOaJi+apnyBPZmZsb2FkTWFuYWdlciDnlKjvvInjgIIKICAgICAgICAgICAgbiA9IDIwMF8wMDAKICAgICAgICAgICAgaWYgImJ5dGVzPSIgaW4gc2VsZi5wYXRoOgogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIG4gPSBtYXgoMSwgbWluKGludChzZWxmLnBhdGguc3BsaXQoImJ5dGVzPSIpWzFdLnNwbGl0KCImIilbMF0pLCA1XzAwMF8wMDApKQogICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjogICMgbm9xYTogQkxFMDAxCiAgICAgICAgICAgICAgICAgICAgbiA9IDIwMF8wMDAKICAgICAgICAgICAgc2VsZi5zZW5kX3Jlc3BvbnNlKDIwMCkKICAgICAgICAgICAgc2VsZi5zZW5kX2hlYWRlcigiQ29udGVudC1UeXBlIiwgImFwcGxpY2F0aW9uL29jdGV0LXN0cmVhbSIpCiAgICAgICAgICAgIHNlbGYuc2VuZF9oZWFkZXIoIkNvbnRlbnQtTGVuZ3RoIiwgc3RyKG4pKQogICAgICAgICAgICBzZWxmLmVuZF9oZWFkZXJzKCkKICAgICAgICAgICAgc2VsZi53ZmlsZS53cml0ZShiIlwwIiAqIG4pCiAgICAgICAgICAgIHJldHVybgogICAgICAgIGlmIHNlbGYucGF0aC5zdGFydHN3aXRoKCIvaGVhbHRoIik6CiAgICAgICAgICAgIHNlbGYuX2pzb24oMjAwLCB7CiAgICAgICAgICAgICAgICAib2siOiBUcnVlLAogICAgICAgICAgICAgICAgInJlYWR5IjogX3BpcGUgaXMgbm90IE5vbmUsCiAgICAgICAgICAgICAgICAibW9kZWwiOiBNT0RFTCwKICAgICAgICAgICAgICAgICJkZXZpY2UiOiBfZGV2aWNlLAogICAgICAgICAgICAgICAgImVycm9yIjogX2xvYWRfZXJyLAogICAgICAgICAgICAgICAgImJ1c3kiOiBfYnVzeSwKICAgICAgICAgICAgICAgICJnZW5lcmF0aW5nIjogX2J1c3kgPiAwLAogICAgICAgICAgICAgICAgImRvbmUiOiBfZG9uZSwKICAgICAgICAgICAgICAgICJsYXN0X3Byb21wdCI6IF9sYXN0X3Byb21wdCwKICAgICAgICAgICAgfSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBzZWxmLl9qc29uKDQwNCwgeyJvayI6IEZhbHNlfSkKCiAgICBkZWYgZG9fUE9TVChzZWxmKSAtPiBOb25lOgogICAgICAgIGlmIG5vdCBzZWxmLnBhdGguc3RhcnRzd2l0aCgiL2dlbmVyYXRlIik6CiAgICAgICAgICAgIHNlbGYuX2pzb24oNDA0LCB7Im9rIjogRmFsc2V9KQogICAgICAgICAgICByZXR1cm4KICAgICAgICBpZiBfcGlwZSBpcyBOb25lOgogICAgICAgICAgICBzZWxmLl9qc29uKDUwMywgeyJvayI6IEZhbHNlLCAiZXJyb3IiOiBfbG9hZF9lcnIgb3IgIuaooeWei+Wwmuacqui8ieWFpeWujOaIkO+8jOiri+eojeWAmeWGjeippiJ9KQogICAgICAgICAgICByZXR1cm4KICAgICAgICB0cnk6CiAgICAgICAgICAgIG4gPSBpbnQoc2VsZi5oZWFkZXJzLmdldCgiQ29udGVudC1MZW5ndGgiLCAiMCIpIG9yICIwIikKICAgICAgICAgICAgYm9keSA9IGpzb24ubG9hZHMoc2VsZi5yZmlsZS5yZWFkKG4pIG9yIGIie30iKQogICAgICAgICAgICBwcm9tcHQgPSAoYm9keS5nZXQoInByb21wdCIpIG9yICIiKS5zdHJpcCgpCiAgICAgICAgICAgIHN0ZXBzID0gaW50KGJvZHkuZ2V0KCJzdGVwcyIpIG9yIERFRkFVTFRfU1RFUFMpCiAgICAgICAgICAgIGlmIG5vdCBwcm9tcHQ6CiAgICAgICAgICAgICAgICBzZWxmLl9qc29uKDQwMCwgeyJvayI6IEZhbHNlLCAiZXJyb3IiOiAi57y65bCRIHByb21wdCJ9KQogICAgICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgICAgIHBuZyA9IGdlbmVyYXRlX3BuZyhwcm9tcHQsIHN0ZXBzKQogICAgICAgICAgICBzZWxmLnNlbmRfcmVzcG9uc2UoMjAwKQogICAgICAgICAgICBzZWxmLnNlbmRfaGVhZGVyKCJDb250ZW50LVR5cGUiLCAiaW1hZ2UvcG5nIikKICAgICAgICAgICAgc2VsZi5zZW5kX2hlYWRlcigiQ29udGVudC1MZW5ndGgiLCBzdHIobGVuKHBuZykpKQogICAgICAgICAgICBzZWxmLmVuZF9oZWFkZXJzKCkKICAgICAgICAgICAgc2VsZi53ZmlsZS53cml0ZShwbmcpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6ICAjIG5vcWE6IEJMRTAwMQogICAgICAgICAgICBzZWxmLl9qc29uKDUwMCwgeyJvayI6IEZhbHNlLCAiZXJyb3IiOiBzdHIoZXhjKX0pCgogICAgZGVmIGxvZ19tZXNzYWdlKHNlbGYsIGZtdDogc3RyLCAqYXJncykgLT4gTm9uZToKICAgICAgICBzeXMuc3RkZXJyLndyaXRlKCJbJXNdICVzXG4iICUgKHRpbWUuc3RyZnRpbWUoIiVIOiVNOiVTIiksIGZtdCAlIGFyZ3MpKQoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIHRocmVhZGluZy5UaHJlYWQodGFyZ2V0PWxvYWRfcGlwZSwgZGFlbW9uPVRydWUpLnN0YXJ0KCkKICAgICMg6aCQ6KitIDAuMC4wLjDvvIjmiYvmqZ8gV2ktRmkg6ZyA6KaB77yJ77yb5Y+q55SoIFVTQihhZGIgcmV2ZXJzZSkg55qE5Lq65Y+v6KitIFZPSUNFV0VBVkVSX0JJTkQ9MTI3LjAuMC4xIOmOluacrOapn+OAggogICAgYmluZCA9IG9zLmVudmlyb24uZ2V0KCJWT0lDRVdFQVZFUl9CSU5EIiwgIjAuMC4wLjAiKQogICAgc2VydmVyID0gVGhyZWFkaW5nSFRUUFNlcnZlcigoYmluZCwgUE9SVCksIEhhbmRsZXIpCiAgICBwcmludChmIlZvaWNlV2VhdmVyIGltYWdlIHNlcnZlcjogaHR0cDovL3tiaW5kfTp7UE9SVH0gKG1vZGVsIHtNT0RFTH0pIiwgZmx1c2g9VHJ1ZSkKICAgIHNlcnZlci5zZXJ2ZV9mb3JldmVyKCkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=='))
    procs['image']=subprocess.Popen(['python','/content/img.py'],env=dict(os.environ,VOICEWEAVER_IMAGE_PORT='9881',VOICEWEAVER_BIND='127.0.0.1'),
        stdout=open('/content/img.log','w'),stderr=subprocess.STDOUT)
    _wait('http://127.0.0.1:9881/health','生圖 SD-Turbo')
if RUN_TEXT:
    open('/content/txt.py','wb').write(base64.b64decode('IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJWb2ljZVdlYXZlciDmnKzmqZ/mloflrZfmjqjoq5bkvLrmnI3lmajvvIhNYWMg56uv77yM5YWN6YeR6ZGw77yJ44CCCgrnlbbmiYvmqZ/oiIfpm7vohabpgKPnt5rlpKDlv6vvvIjmnInnt5ogVVNCIOaIlumrmOmAnyBXaS1Gae+8ieaZgu+8jFZvaWNlV2VhdmVyIOacg+aKiuOAjOWPpeWtkOmHjee1hOOAjemAmemghQrmloflrZfpgYvnrpfkuZ/kuqTntabpm7vohabvvIh0aWVyLUZVTEzvvInvvIznm6Hph4/muJvlsJHmiYvmqZ/osqDmk5TjgILpgJnmlK/kvLrmnI3lmajlsLHmmK/pm7vohabnq6/nmoTmloflrZflpKfohabvvJoK55SoIHRyYW5zZm9ybWVycyDlnKggQXBwbGUgTVBTIOS4iui3keS4gOWAi+Wwj+WeiyBpbnN0cnVjdCDmqKHlnovvvIjpoJDoqK0gUXdlbjIuNS0xLjVCLUluc3RydWN077yMCuWkmuiqnu+8muS4re+8j+iLse+8j+aXpe+8j+mfk++8ie+8jOmmluasoeWft+ihjOacg+iHquWLleS4i+i8ie+8iOWFjemHkemRsO+8ieOAggoK56uv6bue77yaCiAgICBHRVQgIC9oZWFsdGggICAgICAgIC0+IHtvaywgcmVhZHksIG1vZGVsLCBkZXZpY2UsIGVycm9yfQogICAgUE9TVCAvcmVjb25zdHJ1Y3QgICBib2R5IHsic3lzdGVtIjogIi4uLiIsICJ1c2VyIjogIi4uLiJ9IC0+IHsidGV4dCI6ICIuLi4ifQoK55Kw5aKD6K6K5pW477yaCiAgICBURVhUX0lORkVSX01PREVM77yI6aCQ6KitIFF3ZW4vUXdlbjIuNS0xLjVCLUluc3RydWN077yb5oOz5pu05b+r5Y+v55SoIFF3ZW4vUXdlbjIuNS0wLjVCLUluc3RydWN077yJCiAgICBWT0lDRVdFQVZFUl9URVhUX1BPUlTvvIjpoJDoqK0gOTg4Mu+8iQoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBzeXMKaW1wb3J0IHRocmVhZGluZwppbXBvcnQgdGltZQpmcm9tIGh0dHAuc2VydmVyIGltcG9ydCBCYXNlSFRUUFJlcXVlc3RIYW5kbGVyLCBUaHJlYWRpbmdIVFRQU2VydmVyCgpQT1JUID0gaW50KG9zLmVudmlyb24uZ2V0KCJWT0lDRVdFQVZFUl9URVhUX1BPUlQiLCAiOTg4MiIpKQpNT0RFTCA9IG9zLmVudmlyb24uZ2V0KCJURVhUX0lORkVSX01PREVMIiwgIlF3ZW4vUXdlbjIuNS0xLjVCLUluc3RydWN0IikKTUFYX05FV19UT0tFTlMgPSBpbnQob3MuZW52aXJvbi5nZXQoIlRFWFRfSU5GRVJfTUFYX1RPS0VOUyIsICI5NiIpKQoKX21vZGVsID0gTm9uZQpfdG9rID0gTm9uZQpfbG9jayA9IHRocmVhZGluZy5Mb2NrKCkgICAgICAgIyDluo/liJfljJbmjqjoq5bvvIzpgb/lhY0gTVBTIOiomOaGtumrlOeIhuaOiQpfbG9hZF9lcnIgPSAiIgpfZGV2aWNlID0gImNwdSIKCgpkZWYgbG9hZF9tb2RlbCgpIC0+IE5vbmU6CiAgICAiIiLog4zmma/ovInlhaXmqKHlnovvvIjpppbmrKHmnIPoh6rli5XkuIvovInvvInvvIzorpMgc2VydmVyIOiDveeri+WIu+WbnuaHiSAvaGVhbHRo44CCIiIiCiAgICBnbG9iYWwgX21vZGVsLCBfdG9rLCBfbG9hZF9lcnIsIF9kZXZpY2UKICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2gKICAgICAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b01vZGVsRm9yQ2F1c2FsTE0sIEF1dG9Ub2tlbml6ZXIKCiAgICAgICAgIyDoo53nva7lhKrlhYjluo/vvJpDVURB77yIV2luZG93cy9MaW51eCBOVklESUHvvInihpIgTVBT77yIQXBwbGXvvInihpIgQ1BV44CCCiAgICAgICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICAgICAgX2RldmljZSA9ICJjdWRhIgogICAgICAgIGVsaWYgZ2V0YXR0cih0b3JjaC5iYWNrZW5kcywgIm1wcyIsIE5vbmUpIGlzIG5vdCBOb25lIGFuZCB0b3JjaC5iYWNrZW5kcy5tcHMuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgICAgIF9kZXZpY2UgPSAibXBzIgogICAgICAgIGVsc2U6CiAgICAgICAgICAgIF9kZXZpY2UgPSAiY3B1IgogICAgICAgIGR0eXBlID0gdG9yY2guZmxvYXQxNiBpZiBfZGV2aWNlIGluICgiY3VkYSIsICJtcHMiKSBlbHNlIHRvcmNoLmZsb2F0MzIKICAgICAgICBfdG9rID0gQXV0b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQoTU9ERUwpCiAgICAgICAgbW9kZWwgPSBBdXRvTW9kZWxGb3JDYXVzYWxMTS5mcm9tX3ByZXRyYWluZWQoTU9ERUwsIHRvcmNoX2R0eXBlPWR0eXBlKQogICAgICAgIG1vZGVsID0gbW9kZWwudG8oX2RldmljZSkKICAgICAgICBtb2RlbC5ldmFsKCkKICAgICAgICBfbW9kZWwgPSBtb2RlbAogICAgICAgIHByaW50KGYiW3RleHRdIG1vZGVsIHJlYWR5OiB7TU9ERUx9IG9uIHtfZGV2aWNlfSIsIGZsdXNoPVRydWUpCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzogICMgbm9xYTogQkxFMDAxCiAgICAgICAgX2xvYWRfZXJyID0gc3RyKGV4YykKICAgICAgICBwcmludChmIlt0ZXh0XSBsb2FkIGZhaWxlZDoge2V4Y30iLCBmbHVzaD1UcnVlKQoKCmRlZiByZWNvbnN0cnVjdChzeXN0ZW06IHN0ciwgdXNlcjogc3RyLCB0ZW1wZXJhdHVyZTogZmxvYXQgPSAwLjUpIC0+IHN0cjoKICAgIGltcG9ydCB0b3JjaAoKICAgICMg6KmV5YiG6aGe5Lu75YuZ5pyD5YKz5L2O5rqr77yIMC4xfjAuMu+8ieaxguepqeWumu+8m+WkvuWcqOWuieWFqOevhOWcjQogICAgdGVtcGVyYXR1cmUgPSBtYXgoMC4wNSwgbWluKDEuNSwgZmxvYXQodGVtcGVyYXR1cmUgb3IgMC41KSkpCiAgICBtZXNzYWdlcyA9IFtdCiAgICBpZiBzeXN0ZW0uc3RyaXAoKToKICAgICAgICBtZXNzYWdlcy5hcHBlbmQoeyJyb2xlIjogInN5c3RlbSIsICJjb250ZW50Ijogc3lzdGVtfSkKICAgIG1lc3NhZ2VzLmFwcGVuZCh7InJvbGUiOiAidXNlciIsICJjb250ZW50IjogdXNlcn0pCiAgICB3aXRoIF9sb2NrOgogICAgICAgIHByb21wdCA9IF90b2suYXBwbHlfY2hhdF90ZW1wbGF0ZSgKICAgICAgICAgICAgbWVzc2FnZXMsIHRva2VuaXplPUZhbHNlLCBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9VHJ1ZQogICAgICAgICkKICAgICAgICBpbnB1dHMgPSBfdG9rKHByb21wdCwgcmV0dXJuX3RlbnNvcnM9InB0IikudG8oX2RldmljZSkKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgb3V0ID0gX21vZGVsLmdlbmVyYXRlKAogICAgICAgICAgICAgICAgKippbnB1dHMsCiAgICAgICAgICAgICAgICBtYXhfbmV3X3Rva2Vucz1NQVhfTkVXX1RPS0VOUywKICAgICAgICAgICAgICAgIGRvX3NhbXBsZT1UcnVlLAogICAgICAgICAgICAgICAgdGVtcGVyYXR1cmU9dGVtcGVyYXR1cmUsCiAgICAgICAgICAgICAgICB0b3BfcD0wLjksCiAgICAgICAgICAgICAgICBwYWRfdG9rZW5faWQ9X3Rvay5lb3NfdG9rZW5faWQsCiAgICAgICAgICAgICkKICAgICAgICBnZW4gPSBvdXRbMF1baW5wdXRzWyJpbnB1dF9pZHMiXS5zaGFwZVsxXTpdCiAgICAgICAgdGV4dCA9IF90b2suZGVjb2RlKGdlbiwgc2tpcF9zcGVjaWFsX3Rva2Vucz1UcnVlKS5zdHJpcCgpCiAgICB0ZXh0ID0gKHRleHQgb3IgIiIpLnN0cmlwKCkKICAgICMgSlNPTiDovLjlh7rvvIjlvqnlgaXoqZXliIbnrYnntZDmp4vljJbku7vli5nvvInlv4XpoIjlrozmlbTkv53nlZnigJTigJTlpJrooYwgSlNPTiDnoI3liLDnrKzkuIDooYzmnIPmlbTljIXlo57mjonvvIwKICAgICMg5a6i5oi256uv6Kej5p6Q5aSx5pWX5bCx5oKE5oKE5o6J5Yiw5Yil55qE6KmV5YiG5L6G5rqQ77yM5YiG5pW45b+96auY5b+95L2O44CCCiAgICAjIOaooeWei+W4uOWMhSBgYGBqc29uIOWcjeashOaIluWKoOWJjeiogO+8jOaJgOS7peS4jeiDveWPqueci+mWi+mgre+8muWPluesrOS4gOWAiyB7IOWIsOacgOW+jOS4gOWAiyB977yM6IO9IGpzb24g6Kej5p6Q5omN566X44CCCiAgICBpZiAieyIgaW4gdGV4dDoKICAgICAgICBzaSwgZWkgPSB0ZXh0LmZpbmQoInsiKSwgdGV4dC5yZmluZCgifSIpCiAgICAgICAgaWYgZWkgPiBzaToKICAgICAgICAgICAgY2FuZCA9IHRleHRbc2k6ZWkgKyAxXQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBqc29uLmxvYWRzKGNhbmQpCiAgICAgICAgICAgICAgICByZXR1cm4gY2FuZAogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgcGFzcwogICAgIyDlhbbppJjvvIhBQUMg57WE5Y+l562J44CM5LiA5Y+l6Kmx44CN6ZyA5rGC77yJ57at5oyB5Y+q5Y+W56ys5LiA6KGM44CB5Y675byV6JmfCiAgICB0ZXh0ID0gdGV4dC5zcGxpdGxpbmVzKClbMF0uc3RyaXAoKSBpZiB0ZXh0IGVsc2UgIiIKICAgIHJldHVybiB0ZXh0LnN0cmlwKCkuc3RyaXAoIuOAjOOAjVwiJyIpCgoKY2xhc3MgSGFuZGxlcihCYXNlSFRUUFJlcXVlc3RIYW5kbGVyKToKICAgIGRlZiBfanNvbihzZWxmLCBjb2RlOiBpbnQsIG9iajogZGljdCkgLT4gTm9uZToKICAgICAgICBiID0ganNvbi5kdW1wcyhvYmosIGVuc3VyZV9hc2NpaT1GYWxzZSkuZW5jb2RlKCJ1dGYtOCIpCiAgICAgICAgc2VsZi5zZW5kX3Jlc3BvbnNlKGNvZGUpCiAgICAgICAgc2VsZi5zZW5kX2hlYWRlcigiQ29udGVudC1UeXBlIiwgImFwcGxpY2F0aW9uL2pzb247IGNoYXJzZXQ9dXRmLTgiKQogICAgICAgIHNlbGYuc2VuZF9oZWFkZXIoIkNvbnRlbnQtTGVuZ3RoIiwgc3RyKGxlbihiKSkpCiAgICAgICAgc2VsZi5lbmRfaGVhZGVycygpCiAgICAgICAgc2VsZi53ZmlsZS53cml0ZShiKQoKICAgIGRlZiBkb19HRVQoc2VsZikgLT4gTm9uZToKICAgICAgICBpZiBzZWxmLnBhdGguc3RhcnRzd2l0aCgiL2hlYWx0aCIpOgogICAgICAgICAgICBzZWxmLl9qc29uKDIwMCwgewogICAgICAgICAgICAgICAgIm9rIjogVHJ1ZSwKICAgICAgICAgICAgICAgICJyZWFkeSI6IF9tb2RlbCBpcyBub3QgTm9uZSwKICAgICAgICAgICAgICAgICJtb2RlbCI6IE1PREVMLAogICAgICAgICAgICAgICAgImRldmljZSI6IF9kZXZpY2UsCiAgICAgICAgICAgICAgICAiZXJyb3IiOiBfbG9hZF9lcnIsCiAgICAgICAgICAgIH0pCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2VsZi5fanNvbig0MDQsIHsib2siOiBGYWxzZX0pCgogICAgZGVmIGRvX1BPU1Qoc2VsZikgLT4gTm9uZToKICAgICAgICBpZiBub3Qgc2VsZi5wYXRoLnN0YXJ0c3dpdGgoIi9yZWNvbnN0cnVjdCIpOgogICAgICAgICAgICBzZWxmLl9qc29uKDQwNCwgeyJvayI6IEZhbHNlfSkKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgaWYgX21vZGVsIGlzIE5vbmU6CiAgICAgICAgICAgIHNlbGYuX2pzb24oNTAzLCB7Im9rIjogRmFsc2UsICJlcnJvciI6IF9sb2FkX2VyciBvciAi5qih5Z6L5bCa5pyq6LyJ5YWl5a6M5oiQ77yM6KuL56iN5YCZ5YaN6KmmIn0pCiAgICAgICAgICAgIHJldHVybgogICAgICAgIHRyeToKICAgICAgICAgICAgbiA9IGludChzZWxmLmhlYWRlcnMuZ2V0KCJDb250ZW50LUxlbmd0aCIsICIwIikgb3IgIjAiKQogICAgICAgICAgICBib2R5ID0ganNvbi5sb2FkcyhzZWxmLnJmaWxlLnJlYWQobikgb3IgYiJ7fSIpCiAgICAgICAgICAgIHN5c3RlbSA9IChib2R5LmdldCgic3lzdGVtIikgb3IgIiIpLnN0cmlwKCkKICAgICAgICAgICAgdXNlciA9IChib2R5LmdldCgidXNlciIpIG9yICIiKS5zdHJpcCgpCiAgICAgICAgICAgIGlmIG5vdCB1c2VyOgogICAgICAgICAgICAgICAgc2VsZi5fanNvbig0MDAsIHsib2siOiBGYWxzZSwgImVycm9yIjogIue8uuWwkSB1c2VyIn0pCiAgICAgICAgICAgICAgICByZXR1cm4KICAgICAgICAgICAgdGVtcGVyYXR1cmUgPSBib2R5LmdldCgidGVtcGVyYXR1cmUiLCAwLjUpCiAgICAgICAgICAgIHRleHQgPSByZWNvbnN0cnVjdChzeXN0ZW0sIHVzZXIsIHRlbXBlcmF0dXJlKQogICAgICAgICAgICBzZWxmLl9qc29uKDIwMCwgeyJvayI6IFRydWUsICJ0ZXh0IjogdGV4dH0pCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6ICAjIG5vcWE6IEJMRTAwMQogICAgICAgICAgICBzZWxmLl9qc29uKDUwMCwgeyJvayI6IEZhbHNlLCAiZXJyb3IiOiBzdHIoZXhjKX0pCgogICAgZGVmIGxvZ19tZXNzYWdlKHNlbGYsIGZtdDogc3RyLCAqYXJncykgLT4gTm9uZToKICAgICAgICBzeXMuc3RkZXJyLndyaXRlKCJbJXNdICVzXG4iICUgKHRpbWUuc3RyZnRpbWUoIiVIOiVNOiVTIiksIGZtdCAlIGFyZ3MpKQoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIHRocmVhZGluZy5UaHJlYWQodGFyZ2V0PWxvYWRfbW9kZWwsIGRhZW1vbj1UcnVlKS5zdGFydCgpCiAgICAjIOmgkOiorSAwLjAuMC4w77yI5omL5qmfIFdpLUZpIOmcgOimge+8ie+8m+WPqueUqCBVU0IoYWRiIHJldmVyc2UpIOeahOS6uuWPr+iorSBWT0lDRVdFQVZFUl9CSU5EPTEyNy4wLjAuMSDpjpbmnKzmqZ/jgIIKICAgIGJpbmQgPSBvcy5lbnZpcm9uLmdldCgiVk9JQ0VXRUFWRVJfQklORCIsICIwLjAuMC4wIikKICAgIHNlcnZlciA9IFRocmVhZGluZ0hUVFBTZXJ2ZXIoKGJpbmQsIFBPUlQpLCBIYW5kbGVyKQogICAgcHJpbnQoZiJWb2ljZVdlYXZlciB0ZXh0IHNlcnZlcjogaHR0cDovL3tiaW5kfTp7UE9SVH0gKG1vZGVsIHtNT0RFTH0pIiwgZmx1c2g9VHJ1ZSkKICAgIHNlcnZlci5zZXJ2ZV9mb3JldmVyKCkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=='))
    procs['text']=subprocess.Popen(['python','/content/txt.py'],env=dict(os.environ,VOICEWEAVER_TEXT_PORT='9882',VOICEWEAVER_BIND='127.0.0.1'),
        stdout=open('/content/txt.log','w'),stderr=subprocess.STDOUT)
    _wait('http://127.0.0.1:9882/health','文字 Qwen')
print('服務啟動完成')


## ⑥ 啟動雲端橋接（:9879，app + web 共用，含 CORS、動態偵測三功能）


In [ ]:
open('/content/bridge.py','wb').write(base64.b64decode('IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJWb2ljZVdlYXZlciDpm7Lnq6/mqYvmjqXvvIhDb2xhYiAvIOS7u+S9leaciSBHUFUg55qE5qmf5Zmo77yJ44CCCgrkuIDlgIvln6DlkIzmmYLmnI3li5nlhannqK7lrqLmiLbnq6/vvIzlhbHnlKjkuIDmop0gQ2xvdWRmbGFyZS9uZ3JvayDpgJrpgZPvvJoKCiAg4pSA4pSAIEFwcO+8iEFuZHJvaWTvvInilIDilIAg55u05o6l6KybIEdQVC1Tb1ZJVFMgYXBpX3YyIOWNlOWumu+8jOacrOapi+aOpeOAjOWPjeWQkeS7o+eQhuOAjeWIsCAxMjcuMC4wLjE6OTg4MO+8mgogICAgICBHRVQgIC9oZWFsdGggICAgICAgICAgICAgICAgICAgIO+8iGFwcCDlj6rnnIsgSFRUUCAyMDAg5Yik5rS777yJCiAgICAgIEdFVCAgL21vZGVscyAgICAgICAgICAgICAgICAgICAg6KeS6Imy5riF5Zau77yIYXBwIOWLleaFi+mhr+ekuuOAgeWIh+aPm+eUqO+8iQogICAgICBHRVQgIC9zZXRfZ3B0X3dlaWdodHM/Li4uICAgICAgIOWIhyBHUFQg5qyK6YeNCiAgICAgIEdFVCAgL3NldF9zb3ZpdHNfd2VpZ2h0cz8uLi4gICAg5YiHIFNvVklUUyDmrIrph40KICAgICAgR0VUfFBPU1QgL3R0cyAgICAgICAgICAgICAgICAgICDlkIjmiJAKICAgICAgR0VUICAvY29udHJvbCAvc3RvcCAvc2V0X3JlZmVyX2F1ZGlvIOKApu+8iOWFtumkmCBhcGlfdjIg6Lev55Sx5YWo6YCP5YKz77yJCgogIOKUgOKUgCBXZWLvvIhnaXRodWIuaW/vvIzngI/opr3lmajvvInilIDilIAg6Kyb6Kqe6Z+z5Lit5b+DIDk4Nzkg5qmL5o6l5Y2U5a6a77yM6ZyA6KaBIENPUlPvvJoKICAgICAgR0VUICAvaGVhbHRoICAg4oaSIHtvaywgdm9pY2UsIGltYWdlLCB0ZXh0LCBjdXJyZW50LCBsYW5nfQogICAgICBHRVQgIC92b2ljZXMgICDihpIge29rLCB2b2ljZXM6W3tuYW1lLCBsYW5nfV19CiAgICAgIFBPU1QgL3N3aXRjaCAgIHtuYW1lLCBsYW5nfSAgICAgICAg6LyJ5bCN5oeJ5qyK6YeNCiAgICAgIFBPU1QgL3NwZWFrICAgIHt0ZXh0LCBuYW1lPywgbGFuZz99IOKGkiBhdWRpby93YXYKCueSsOWig+iuiuaVuO+8mgogIEdTVl9ST09UICAgICAgR1BULVNvVklUUyDlsIjmoYjmoLnnm67pjITvvIhncHRfcmVsL3Nvdml0c19yZWwg55u45bCN5pa85q2k77yJCiAgSElTVE9SWV9QQVRIICBtb2RlbF9oaXN0b3J5Lmpzb24g6Lev5b6R77yI6aCQ6KitICRHU1ZfUk9PVC91c2VyX21vZGVscy9tb2RlbF9oaXN0b3J5Lmpzb27vvIkKICBBUElfQkFTRSAgICAgIGFwaV92MiDkvY3lnYDvvIjpoJDoqK0gaHR0cDovLzEyNy4wLjAuMTo5ODgw77yJCiAgQlJJREdFX1BPUlQgICDmnKzmqYvmjqXln6DvvIjpoJDoqK0gOTg3Oe+8iQoiIiIKaW1wb3J0IGh0dHAuc2VydmVyCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgcmUKaW1wb3J0IHNvY2tldHNlcnZlcgppbXBvcnQgdGhyZWFkaW5nCmltcG9ydCB1cmxsaWIucGFyc2UKaW1wb3J0IHVybGxpYi5yZXF1ZXN0CgpHU1ZfUk9PVCA9IG9zLmVudmlyb24uZ2V0KCJHU1ZfUk9PVCIsICIuIikucnN0cmlwKCIvIikKSElTVE9SWV9QQVRIID0gb3MuZW52aXJvbi5nZXQoIkhJU1RPUllfUEFUSCIsIGYie0dTVl9ST09UfS91c2VyX21vZGVscy9tb2RlbF9oaXN0b3J5Lmpzb24iKQpBUElfQkFTRSA9IG9zLmVudmlyb24uZ2V0KCJBUElfQkFTRSIsICJodHRwOi8vMTI3LjAuMC4xOjk4ODAiKS5yc3RyaXAoIi8iKQpCUklER0VfUE9SVCA9IGludChvcy5lbnZpcm9uLmdldCgiQlJJREdFX1BPUlQiLCAiOTg3OSIpKQpJTUFHRV9CQVNFID0gb3MuZW52aXJvbi5nZXQoIklNQUdFX0JBU0UiLCAiaHR0cDovLzEyNy4wLjAuMTo5ODgxIikucnN0cmlwKCIvIikgICAjIFNELVR1cmJvIOeUn+WclgpURVhUX0JBU0UgPSBvcy5lbnZpcm9uLmdldCgiVEVYVF9CQVNFIiwgImh0dHA6Ly8xMjcuMC4wLjE6OTg4MiIpLnJzdHJpcCgiLyIpICAgICAjIFF3ZW4g5paH5a2XL+ipleWIhgoKCmRlZiBfcG9ydF91cChiYXNlKToKICAgICIiIuaOoua4rOafkOacrOapn+acjeWLmeaYr+WQpuWcqOe3mu+8iOe1piAvaGVhbHRoIOWLleaFi+WbnuWgsSB2b2ljZS9pbWFnZS90ZXh0IOS4iemgheWPr+eUqOaAp++8ieOAgiIiIgogICAgaW1wb3J0IHNvY2tldAogICAgdHJ5OgogICAgICAgIHUgPSB1cmxsaWIucGFyc2UudXJscGFyc2UoYmFzZSkKICAgICAgICB3aXRoIHNvY2tldC5jcmVhdGVfY29ubmVjdGlvbigodS5ob3N0bmFtZSwgdS5wb3J0IG9yIDgwKSwgdGltZW91dD0wLjYpOgogICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gRmFsc2UKCiMgYXBpX3YyIOWOn+eUn+i3r+eUsSDihpIg5LiA5b6L6YCP5YKz77yIYXBwIOi1sOmAmeS6m++8ieOAgi9tb2RlbHMg5LiN5Zyo5q2k5YiX77ya5a6Y5pa5IGFwaV92MiDmspLmnInpgJnot6/nlLHvvIwKIyDnlLHmnKzmqYvmjqXnm7TmjqXoroAgbW9kZWxfaGlzdG9yeS5qc29uIOeUoueUn++8iGFwcCDoiIcgd2ViIOWFseeUqOWQjOS4gOS7vea4heWWru+8ieOAggpQUk9YWV9QUkVGSVhFUyA9ICgiL3R0cyIsICIvc2V0X2dwdF93ZWlnaHRzIiwgIi9zZXRfc292aXRzX3dlaWdodHMiLAogICAgICAgICAgICAgICAgICAiL3NldF9yZWZlcl9hdWRpbyIsICIvY29udHJvbCIsICIvc3RvcCIpCgpfTEFOR19UQUdfVE9fQVBJID0geyJaSCI6ICJ6aCIsICJFTiI6ICJlbiIsICJKQSI6ICJqYSIsICJLTyI6ICJrbyJ9Cl9sb2NrID0gdGhyZWFkaW5nLkxvY2soKQpfY3VycmVudCA9IHsibmFtZSI6ICIiLCAibGFuZyI6ICIifSAgICMg55uu5YmN6LyJ5YWl55qE6KeS6Imy77yI5oOw5oCn5YiH5o+b55So77yJCgoKZGVmIF9sb2FkX2hpc3RvcnkoKToKICAgIHRyeToKICAgICAgICB3aXRoIG9wZW4oSElTVE9SWV9QQVRILCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgICAgICBkYXRhID0ganNvbi5sb2FkKGYpCiAgICAgICAgcmV0dXJuIGRhdGEgaWYgaXNpbnN0YW5jZShkYXRhLCBsaXN0KSBlbHNlIFtdCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiBbXQoKCmRlZiBfbGFuZ19vZihyZWMpOgogICAgIiIi5b6e6KeS6Imy5ZCN5bC+57a05o6o6Kqe6KiA5qiZ57Gk77yIWkgvRU4vSkEvS0/vvInvvJvmspLmnInlsLEgWkjjgIIiIiIKICAgIG5hbWUgPSBzdHIocmVjLmdldCgibmFtZSIsICIiKSkKICAgIG0gPSByZS5zZWFyY2gociJbIF9dKFpIfENOfEVOfEpBfEpQfEtPfEtSKVxiIiwgbmFtZSwgcmUuSUdOT1JFQ0FTRSkKICAgIGlmIG5vdCBtOgogICAgICAgIHJldHVybiAiWkgiCiAgICB0ID0gbS5ncm91cCgxKS51cHBlcigpCiAgICByZXR1cm4geyJDTiI6ICJaSCIsICJKUCI6ICJKQSIsICJLUiI6ICJLTyJ9LmdldCh0LCB0KQoKCmRlZiBfZmluZChuYW1lLCBsYW5nPSIiKToKICAgIGhpc3QgPSBfbG9hZF9oaXN0b3J5KCkKICAgIHdhbnQgPSAobGFuZyBvciAiIikudXBwZXIoKS5yZXBsYWNlKCJDTiIsICJaSCIpLnJlcGxhY2UoIkpQIiwgIkpBIikucmVwbGFjZSgiS1IiLCAiS08iKQogICAgY2FuZHMgPSBbciBmb3IgciBpbiBoaXN0IGlmIHIuZ2V0KCJuYW1lIikgPT0gbmFtZV0KICAgIGlmIHdhbnQ6CiAgICAgICAgZXhhY3QgPSBbciBmb3IgciBpbiBjYW5kcyBpZiBfbGFuZ19vZihyKSA9PSB3YW50XQogICAgICAgIGlmIGV4YWN0OgogICAgICAgICAgICByZXR1cm4gZXhhY3RbMF0KICAgIHJldHVybiBjYW5kc1swXSBpZiBjYW5kcyBlbHNlIE5vbmUKCgpkZWYgX2FwaV9nZXQocGF0aCk6CiAgICB3aXRoIHVybGxpYi5yZXF1ZXN0LnVybG9wZW4oZiJ7QVBJX0JBU0V9e3BhdGh9IiwgdGltZW91dD0xMjApIGFzIHI6CiAgICAgICAgcmV0dXJuIHIuc3RhdHVzLCByLnJlYWQoKQoKCmRlZiBfc3dpdGNoKG5hbWUsIGxhbmcpOgogICAgIiIi6LyJ5YWl5oyH5a6a6KeS6Imy55qEIEdQVCArIFNvVklUUyDmrIrph43vvIjmg7DmgKfvvJroiIfnm67liY3nm7jlkIzlsLHot7PpgY7vvInjgILlm54gKG9rLCBtc2cp44CCIiIiCiAgICB3aXRoIF9sb2NrOgogICAgICAgIGlmIF9jdXJyZW50WyJuYW1lIl0gPT0gbmFtZSBhbmQgKG5vdCBsYW5nIG9yIF9jdXJyZW50WyJsYW5nIl0gPT0gX2xhbmdfb2YoeyJuYW1lIjogZiJ4IHtsYW5nfSJ9KSk6CiAgICAgICAgICAgIHJldHVybiBUcnVlLCAiYWxyZWFkeSBsb2FkZWQiCiAgICAgICAgcmVjID0gX2ZpbmQobmFtZSwgbGFuZykKICAgICAgICBpZiBub3QgcmVjOgogICAgICAgICAgICByZXR1cm4gRmFsc2UsIGYi5om+5LiN5Yiw6KeS6Imy77yae25hbWV9IgogICAgICAgIGdwdCA9IHJlYy5nZXQoImdwdF9yZWwiLCAiIikKICAgICAgICBzb3ZpdHMgPSByZWMuZ2V0KCJzb3ZpdHNfcmVsIiwgIiIpCiAgICAgICAgaWYgbm90IGdwdCBvciBub3Qgc292aXRzOgogICAgICAgICAgICByZXR1cm4gRmFsc2UsIGYi6KeS6ImyIHtuYW1lfSDnvLrlsJHmrIrph43ot6/lvpEiCiAgICAgICAgdHJ5OgogICAgICAgICAgICBmb3IgZXAsIHdwIGluICgoIi9zZXRfc292aXRzX3dlaWdodHMiLCBzb3ZpdHMpLCAoIi9zZXRfZ3B0X3dlaWdodHMiLCBncHQpKToKICAgICAgICAgICAgICAgIHEgPSB1cmxsaWIucGFyc2UudXJsZW5jb2RlKHsid2VpZ2h0c19wYXRoIjogd3B9KQogICAgICAgICAgICAgICAgY29kZSwgYm9keSA9IF9hcGlfZ2V0KGYie2VwfT97cX0iKQogICAgICAgICAgICAgICAgaWYgY29kZSAhPSAyMDA6CiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIEZhbHNlLCBmIntlcH0gSFRUUCB7Y29kZX06IHtib2R5WzoxNjBdLmRlY29kZSgndXRmLTgnLCdyZXBsYWNlJyl9IgogICAgICAgICAgICBfY3VycmVudFsibmFtZSJdID0gbmFtZQogICAgICAgICAgICBfY3VycmVudFsibGFuZyJdID0gX2xhbmdfb2YocmVjKQogICAgICAgICAgICByZXR1cm4gVHJ1ZSwgInN3aXRjaGVkIgogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgcmV0dXJuIEZhbHNlLCBzdHIoZSkKCgpkZWYgX3JlbWFwX3JlZihyZWZfcGF0aCk6CiAgICAiIiJtb2RlbF9oaXN0b3J5IOeahCByZWZfcGF0aCDmmK/ljp/mqZ/lmajntZXlsI3ot6/lvpHvvJvlnKjmnKzmqZ/kuI3lrZjlnKjmmYLvvIwKICAgIOWYl+ippuS7peaqlOWQjeWcqCBHU1ZfUk9PVC9yZWZfYXVkaW8g5bqV5LiL6YGe6L+05om+5Zue44CCIiIiCiAgICBpZiByZWZfcGF0aCBhbmQgb3MucGF0aC5leGlzdHMocmVmX3BhdGgpOgogICAgICAgIHJldHVybiByZWZfcGF0aAogICAgYmFzZSA9IG9zLnBhdGguYmFzZW5hbWUocmVmX3BhdGggb3IgIiIpCiAgICBpZiBub3QgYmFzZToKICAgICAgICByZXR1cm4gcmVmX3BhdGgKICAgIGZvciByb290LCBfZGlycywgZmlsZXMgaW4gb3Mud2Fsayhvcy5wYXRoLmpvaW4oR1NWX1JPT1QsICJyZWZfYXVkaW8iKSk6CiAgICAgICAgaWYgYmFzZSBpbiBmaWxlczoKICAgICAgICAgICAgcmV0dXJuIG9zLnBhdGguam9pbihyb290LCBiYXNlKQogICAgcmV0dXJuIHJlZl9wYXRoCgoKZGVmIF9zcGVhayh0ZXh0LCBuYW1lLCBsYW5nKToKICAgICIiIuWQiOaIkOS4gOautSDihpIg5ZueICh3YXZfYnl0ZXMsIGVycinjgILmjIflrprop5LoibLmnIPlhYjmg7DmgKfliIfmj5vjgIIiIiIKICAgIGlmIG5hbWU6CiAgICAgICAgb2ssIG1zZyA9IF9zd2l0Y2gobmFtZSwgbGFuZykKICAgICAgICBpZiBub3Qgb2s6CiAgICAgICAgICAgIHJldHVybiBOb25lLCBtc2cKICAgIGVsaWYgbm90IF9jdXJyZW50WyJuYW1lIl06CiAgICAgICAgaGlzdCA9IF9sb2FkX2hpc3RvcnkoKQogICAgICAgIGlmIG5vdCBoaXN0OgogICAgICAgICAgICByZXR1cm4gTm9uZSwgIumbsuerr+S4iuaykuacieS7u+S9leiqnumfs+aooeWeiyIKICAgICAgICBvaywgbXNnID0gX3N3aXRjaChoaXN0WzBdLmdldCgibmFtZSIsICIiKSwgX2xhbmdfb2YoaGlzdFswXSkpCiAgICAgICAgaWYgbm90IG9rOgogICAgICAgICAgICByZXR1cm4gTm9uZSwgbXNnCiAgICByZWMgPSBfZmluZChfY3VycmVudFsibmFtZSJdKSBvciB7fQogICAgdGFnID0gKGxhbmcgb3IgX2N1cnJlbnRbImxhbmciXSBvciAiWkgiKS51cHBlcigpCiAgICBhcGlfbGFuZyA9IF9MQU5HX1RBR19UT19BUEkuZ2V0KHRhZywgInpoIikKICAgIHBheWxvYWQgPSB7CiAgICAgICAgInRleHQiOiB0ZXh0LCAidGV4dF9sYW5nIjogYXBpX2xhbmcsCiAgICAgICAgInJlZl9hdWRpb19wYXRoIjogX3JlbWFwX3JlZihyZWMuZ2V0KCJyZWZfcGF0aCIsICIiKSksCiAgICAgICAgInByb21wdF90ZXh0IjogcmVjLmdldCgicHJvbXB0IiwgIiIpLCAicHJvbXB0X2xhbmciOiBhcGlfbGFuZywKICAgICAgICAibWVkaWFfdHlwZSI6ICJ3YXYiLCAic3RyZWFtaW5nX21vZGUiOiBGYWxzZSwgInRleHRfc3BsaXRfbWV0aG9kIjogImN1dDUiLAogICAgICAgICJiYXRjaF9zaXplIjogMSwgInRvcF9rIjogMjAsICJ0b3BfcCI6IDAuOTUsICJ0ZW1wZXJhdHVyZSI6IDEuMDUsCiAgICAgICAgInJlcGV0aXRpb25fcGVuYWx0eSI6IDEuMTUsICJmcmFnbWVudF9pbnRlcnZhbCI6IDAuMTgsCiAgICAgICAgInBhcmFsbGVsX2luZmVyIjogVHJ1ZSwgInNwbGl0X2J1Y2tldCI6IEZhbHNlLCAic2FtcGxlX3N0ZXBzIjogMTYsCiAgICAgICAgInNwZWVkX2ZhY3RvciI6IDEuMDIsCiAgICB9CiAgICB0cnk6CiAgICAgICAgcmVxID0gdXJsbGliLnJlcXVlc3QuUmVxdWVzdCgKICAgICAgICAgICAgZiJ7QVBJX0JBU0V9L3R0cyIsIGRhdGE9anNvbi5kdW1wcyhwYXlsb2FkKS5lbmNvZGUoInV0Zi04IiksCiAgICAgICAgICAgIGhlYWRlcnM9eyJDb250ZW50LVR5cGUiOiAiYXBwbGljYXRpb24vanNvbiJ9LCBtZXRob2Q9IlBPU1QiKQogICAgICAgIHdpdGggdXJsbGliLnJlcXVlc3QudXJsb3BlbihyZXEsIHRpbWVvdXQ9MzAwKSBhcyByOgogICAgICAgICAgICBkYXRhID0gci5yZWFkKCkKICAgICAgICBpZiBsZW4oZGF0YSkgPCAxMDI0OgogICAgICAgICAgICByZXR1cm4gTm9uZSwgZGF0YS5kZWNvZGUoInV0Zi04IiwgInJlcGxhY2UiKQogICAgICAgIHJldHVybiBkYXRhLCBOb25lCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgcmV0dXJuIE5vbmUsIHN0cihlKQoKCmNsYXNzIEhhbmRsZXIoaHR0cC5zZXJ2ZXIuQmFzZUhUVFBSZXF1ZXN0SGFuZGxlcik6CiAgICBwcm90b2NvbF92ZXJzaW9uID0gIkhUVFAvMS4xIgoKICAgIGRlZiBsb2dfbWVzc2FnZShzZWxmLCAqYSk6CiAgICAgICAgcGFzcwoKICAgIGRlZiBfY29ycyhzZWxmKToKICAgICAgICBzZWxmLnNlbmRfaGVhZGVyKCJBY2Nlc3MtQ29udHJvbC1BbGxvdy1PcmlnaW4iLCAiKiIpCiAgICAgICAgc2VsZi5zZW5kX2hlYWRlcigiQWNjZXNzLUNvbnRyb2wtQWxsb3ctTWV0aG9kcyIsICJHRVQsIFBPU1QsIE9QVElPTlMiKQogICAgICAgIHNlbGYuc2VuZF9oZWFkZXIoIkFjY2Vzcy1Db250cm9sLUFsbG93LUhlYWRlcnMiLCAiQ29udGVudC1UeXBlIikKCiAgICBkZWYgX2pzb24oc2VsZiwgY29kZSwgb2JqKToKICAgICAgICBiID0ganNvbi5kdW1wcyhvYmosIGVuc3VyZV9hc2NpaT1GYWxzZSkuZW5jb2RlKCJ1dGYtOCIpCiAgICAgICAgc2VsZi5zZW5kX3Jlc3BvbnNlKGNvZGUpOyBzZWxmLl9jb3JzKCkKICAgICAgICBzZWxmLnNlbmRfaGVhZGVyKCJDb250ZW50LVR5cGUiLCAiYXBwbGljYXRpb24vanNvbjsgY2hhcnNldD11dGYtOCIpCiAgICAgICAgc2VsZi5zZW5kX2hlYWRlcigiQ29udGVudC1MZW5ndGgiLCBzdHIobGVuKGIpKSkKICAgICAgICBzZWxmLmVuZF9oZWFkZXJzKCkKICAgICAgICBzZWxmLndmaWxlLndyaXRlKGIpCgogICAgZGVmIF93YXYoc2VsZiwgZGF0YSk6CiAgICAgICAgc2VsZi5zZW5kX3Jlc3BvbnNlKDIwMCk7IHNlbGYuX2NvcnMoKQogICAgICAgIHNlbGYuc2VuZF9oZWFkZXIoIkNvbnRlbnQtVHlwZSIsICJhdWRpby93YXYiKQogICAgICAgIHNlbGYuc2VuZF9oZWFkZXIoIkNvbnRlbnQtTGVuZ3RoIiwgc3RyKGxlbihkYXRhKSkpCiAgICAgICAgc2VsZi5lbmRfaGVhZGVycygpCiAgICAgICAgc2VsZi53ZmlsZS53cml0ZShkYXRhKQoKICAgIGRlZiBfYm9keShzZWxmKToKICAgICAgICBuID0gaW50KHNlbGYuaGVhZGVycy5nZXQoIkNvbnRlbnQtTGVuZ3RoIiwgIjAiKSBvciAiMCIpCiAgICAgICAgdHJ5OgogICAgICAgICAgICByZXR1cm4ganNvbi5sb2FkcyhzZWxmLnJmaWxlLnJlYWQobikgb3IgYiJ7fSIpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgcmV0dXJuIHt9CgogICAgZGVmIF9wcm94eShzZWxmKToKICAgICAgICAiIiLmioogYXBpX3YyIOWOn+eUn+i3r+eUsemAj+WCs+WIsCAxMjcuMC4wLjE6OTg4MO+8iGFwcCDnlKjvvInjgIIiIiIKICAgICAgICB0cnk6CiAgICAgICAgICAgIGNvZGUsIGJvZHkgPSBfYXBpX2dldChzZWxmLnBhdGgpCiAgICAgICAgICAgIHNlbGYuc2VuZF9yZXNwb25zZShjb2RlKTsgc2VsZi5fY29ycygpCiAgICAgICAgICAgIHNlbGYuc2VuZF9oZWFkZXIoIkNvbnRlbnQtVHlwZSIsICJhcHBsaWNhdGlvbi9vY3RldC1zdHJlYW0iKQogICAgICAgICAgICBzZWxmLnNlbmRfaGVhZGVyKCJDb250ZW50LUxlbmd0aCIsIHN0cihsZW4oYm9keSkpKQogICAgICAgICAgICBzZWxmLmVuZF9oZWFkZXJzKCkKICAgICAgICAgICAgc2VsZi53ZmlsZS53cml0ZShib2R5KQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgc2VsZi5fanNvbig1MDIsIHsib2siOiBGYWxzZSwgImVycm9yIjogc3RyKGUpfSkKCiAgICBkZWYgZG9fT1BUSU9OUyhzZWxmKToKICAgICAgICBzZWxmLnNlbmRfcmVzcG9uc2UoMjA0KTsgc2VsZi5fY29ycygpCiAgICAgICAgc2VsZi5zZW5kX2hlYWRlcigiQ29udGVudC1MZW5ndGgiLCAiMCIpOyBzZWxmLmVuZF9oZWFkZXJzKCkKCiAgICBkZWYgZG9fR0VUKHNlbGYpOgogICAgICAgIHBhdGggPSBzZWxmLnBhdGguc3BsaXQoIj8iKVswXQogICAgICAgIGlmIHBhdGggPT0gIi8iIG9yIHBhdGggPT0gIi9oZWFsdGgiOgogICAgICAgICAgICBjdXIgPSBfZmluZChfY3VycmVudFsibmFtZSJdKSBpZiBfY3VycmVudFsibmFtZSJdIGVsc2UgTm9uZQogICAgICAgICAgICByZXR1cm4gc2VsZi5fanNvbigyMDAsIHsKICAgICAgICAgICAgICAgICJvayI6IFRydWUsICJzZXJ2aWNlIjogInZvaWNld2VhdmVyLWNsb3VkLWJyaWRnZSIsCiAgICAgICAgICAgICAgICAiY3VycmVudCI6IF9jdXJyZW50WyJuYW1lIl0sICJsYW5nIjogX2N1cnJlbnRbImxhbmciXSwKICAgICAgICAgICAgICAgICMg5LiJ6aCF6IO95Yqb5YuV5oWL5YG15ris77yabm90ZWJvb2sg5pyJ6LeR5ZOq5YCL5pyN5YuZ5bCx5Zue5aCx5ZOq5YCL54K6IFRydWXvvIjoh6rnlLHpgbjmk4fvvIkKICAgICAgICAgICAgICAgICJ2b2ljZSI6IF9wb3J0X3VwKEFQSV9CQVNFKSwgImltYWdlIjogX3BvcnRfdXAoSU1BR0VfQkFTRSksICJ0ZXh0IjogX3BvcnRfdXAoVEVYVF9CQVNFKSwKICAgICAgICAgICAgICAgICMgYXBwIOmdoCAvaGVhbHRoIGJvZHkg5YWn5ZCr55qE5qyK6YeN5qqU5ZCN5q+U5bCN5Ye644CM55uu5YmN5piv5ZOq5YCL6KeS6Imy44CN4oaSIOiHquWLlemrmOS6rgogICAgICAgICAgICAgICAgInQyc193ZWlnaHRzX3BhdGgiOiAoY3VyIG9yIHt9KS5nZXQoImdwdF9yZWwiLCAiIiksCiAgICAgICAgICAgICAgICAidml0c193ZWlnaHRzX3BhdGgiOiAoY3VyIG9yIHt9KS5nZXQoInNvdml0c19yZWwiLCAiIiksCiAgICAgICAgICAgIH0pCiAgICAgICAgaWYgcGF0aCA9PSAiL3ZvaWNlcyI6CiAgICAgICAgICAgIHZvaWNlcyA9IFt7Im5hbWUiOiByLmdldCgibmFtZSIsICIiKSwgImxhbmciOiBfbGFuZ19vZihyKX0gZm9yIHIgaW4gX2xvYWRfaGlzdG9yeSgpXQogICAgICAgICAgICByZXR1cm4gc2VsZi5fanNvbigyMDAsIHsib2siOiBUcnVlLCAidm9pY2VzIjogdm9pY2VzfSkKICAgICAgICBpZiBwYXRoID09ICIvbW9kZWxzIjoKICAgICAgICAgICAgIyBhcHAg5pyf5pyb55qE5b2i54uA77yI5a6Y5pa5IGFwaV92MiDnhKHmraTot6/nlLHvvIzmnKzmqYvmjqXlvp4gbW9kZWxfaGlzdG9yeS5qc29uIOeUoueUn++8iQogICAgICAgICAgICBvdXQgPSBbeyJuYW1lIjogci5nZXQoIm5hbWUiLCAiIiksICJ2ZXJzaW9uIjogci5nZXQoInZlcnNpb24iLCAiIiksCiAgICAgICAgICAgICAgICAgICAgImdwdF9yZWwiOiByLmdldCgiZ3B0X3JlbCIsICIiKSwgInNvdml0c19yZWwiOiByLmdldCgic292aXRzX3JlbCIsICIiKSwKICAgICAgICAgICAgICAgICAgICAicmVmX3BhdGgiOiByLmdldCgicmVmX3BhdGgiLCAiIiksICJwcm9tcHQiOiByLmdldCgicHJvbXB0IiwgIiIpfQogICAgICAgICAgICAgICAgICAgZm9yIHIgaW4gX2xvYWRfaGlzdG9yeSgpXQogICAgICAgICAgICByZXR1cm4gc2VsZi5fanNvbigyMDAsIHsib2siOiBUcnVlLCAibW9kZWxzIjogb3V0fSkKICAgICAgICBpZiBwYXRoLnN0YXJ0c3dpdGgoUFJPWFlfUFJFRklYRVMpOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fcHJveHkoKQogICAgICAgIHNlbGYuX2pzb24oNDA0LCB7Im9rIjogRmFsc2UsICJlcnJvciI6ICJub3QgZm91bmQifSkKCiAgICBkZWYgZG9fUE9TVChzZWxmKToKICAgICAgICBwYXRoID0gc2VsZi5wYXRoLnNwbGl0KCI/IilbMF0KICAgICAgICBkYXRhID0gc2VsZi5fYm9keSgpCiAgICAgICAgaWYgcGF0aCA9PSAiL3N3aXRjaCI6CiAgICAgICAgICAgIG9rLCBtc2cgPSBfc3dpdGNoKHN0cihkYXRhLmdldCgibmFtZSIsICIiKSksIHN0cihkYXRhLmdldCgibGFuZyIsICIiKSkpCiAgICAgICAgICAgIHJldHVybiBzZWxmLl9qc29uKDIwMCBpZiBvayBlbHNlIDUwMCwgeyJvayI6IG9rLCAibXNnIjogbXNnfSkKICAgICAgICBpZiBwYXRoID09ICIvc3BlYWsiOgogICAgICAgICAgICB0ZXh0ID0gc3RyKGRhdGEuZ2V0KCJ0ZXh0IiwgIiIpKS5zdHJpcCgpCiAgICAgICAgICAgIGlmIG5vdCB0ZXh0OgogICAgICAgICAgICAgICAgcmV0dXJuIHNlbGYuX2pzb24oNDAwLCB7Im9rIjogRmFsc2UsICJlcnJvciI6ICJlbXB0eSB0ZXh0In0pCiAgICAgICAgICAgIHdhdiwgZXJyID0gX3NwZWFrKHRleHQsIHN0cihkYXRhLmdldCgibmFtZSIsICIiKSksIHN0cihkYXRhLmdldCgibGFuZyIsICIiKSkpCiAgICAgICAgICAgIGlmIHdhdiBpcyBOb25lOgogICAgICAgICAgICAgICAgcmV0dXJuIHNlbGYuX2pzb24oNTAwLCB7Im9rIjogRmFsc2UsICJlcnJvciI6IGVycn0pCiAgICAgICAgICAgIHJldHVybiBzZWxmLl93YXYod2F2KQogICAgICAgIGlmIHBhdGggPT0gIi9pbWFnZSI6ICAgICAgICAgICAgICAgICAgIyB3ZWIg55Sf5ZyWIOKGkiDku6PnkIbliLAgU0QtVHVyYm8oOTg4MSnvvIzlm54gUE5HCiAgICAgICAgICAgIHByb21wdCA9IHN0cihkYXRhLmdldCgicHJvbXB0IiwgIiIpKS5zdHJpcCgpCiAgICAgICAgICAgIGlmIG5vdCBwcm9tcHQ6CiAgICAgICAgICAgICAgICByZXR1cm4gc2VsZi5fanNvbig0MDAsIHsib2siOiBGYWxzZSwgImVycm9yIjogImVtcHR5IHByb21wdCJ9KQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBib2R5ID0ganNvbi5kdW1wcyh7InByb21wdCI6IHByb21wdCwgInN0ZXBzIjogaW50KGRhdGEuZ2V0KCJzdGVwcyIsIDIpIG9yIDIpfSkuZW5jb2RlKCkKICAgICAgICAgICAgICAgIHJlcSA9IHVybGxpYi5yZXF1ZXN0LlJlcXVlc3QoZiJ7SU1BR0VfQkFTRX0vZ2VuZXJhdGUiLCBkYXRhPWJvZHksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGhlYWRlcnM9eyJDb250ZW50LVR5cGUiOiAiYXBwbGljYXRpb24vanNvbiJ9LCBtZXRob2Q9IlBPU1QiKQogICAgICAgICAgICAgICAgd2l0aCB1cmxsaWIucmVxdWVzdC51cmxvcGVuKHJlcSwgdGltZW91dD0xODApIGFzIHI6CiAgICAgICAgICAgICAgICAgICAgcG5nID0gci5yZWFkKCkKICAgICAgICAgICAgICAgIHNlbGYuc2VuZF9yZXNwb25zZSgyMDApOyBzZWxmLl9jb3JzKCkKICAgICAgICAgICAgICAgIHNlbGYuc2VuZF9oZWFkZXIoIkNvbnRlbnQtVHlwZSIsICJpbWFnZS9wbmciKQogICAgICAgICAgICAgICAgc2VsZi5zZW5kX2hlYWRlcigiQ29udGVudC1MZW5ndGgiLCBzdHIobGVuKHBuZykpKQogICAgICAgICAgICAgICAgc2VsZi5lbmRfaGVhZGVycygpOyBzZWxmLndmaWxlLndyaXRlKHBuZykKICAgICAgICAgICAgICAgIHJldHVybgogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgICAgICByZXR1cm4gc2VsZi5fanNvbig1MDIsIHsib2siOiBGYWxzZSwgImVycm9yIjogc3RyKGUpfSkKICAgICAgICBpZiBwYXRoID09ICIvdGV4dCI6ICAgICAgICAgICAgICAgICAgICMgd2ViIOaWh+Wtly/oqZXliIYg4oaSIOS7o+eQhuWIsCBRd2VuKDk4ODIpCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIGZ3ZCA9IHsic3lzdGVtIjogZGF0YS5nZXQoInN5c3RlbSIsICIiKSwgInVzZXIiOiBkYXRhLmdldCgidXNlciIsICIiKX0KICAgICAgICAgICAgICAgIGlmICJ0ZW1wZXJhdHVyZSIgaW4gZGF0YToKICAgICAgICAgICAgICAgICAgICBmd2RbInRlbXBlcmF0dXJlIl0gPSBkYXRhWyJ0ZW1wZXJhdHVyZSJdCiAgICAgICAgICAgICAgICByZXEgPSB1cmxsaWIucmVxdWVzdC5SZXF1ZXN0KGYie1RFWFRfQkFTRX0vcmVjb25zdHJ1Y3QiLCBkYXRhPWpzb24uZHVtcHMoZndkKS5lbmNvZGUoKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaGVhZGVycz17IkNvbnRlbnQtVHlwZSI6ICJhcHBsaWNhdGlvbi9qc29uIn0sIG1ldGhvZD0iUE9TVCIpCiAgICAgICAgICAgICAgICB3aXRoIHVybGxpYi5yZXF1ZXN0LnVybG9wZW4ocmVxLCB0aW1lb3V0PTEyMCkgYXMgcjoKICAgICAgICAgICAgICAgICAgICByZXR1cm4gc2VsZi5fanNvbigyMDAsIGpzb24ubG9hZHMoci5yZWFkKCkpKQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgICAgICByZXR1cm4gc2VsZi5fanNvbig1MDIsIHsib2siOiBGYWxzZSwgImVycm9yIjogc3RyKGUpfSkKICAgICAgICBpZiBwYXRoLnN0YXJ0c3dpdGgoIi90dHMiKTogICAgICAgICAgIyBhcGlfdjIgUE9TVCAvdHRz77yIYXBwIOS5n+WPr+iDvei1sCBQT1NU77yJCiAgICAgICAgICAgIHJldHVybiBzZWxmLl9wcm94eV9wb3N0KCkKICAgICAgICBzZWxmLl9qc29uKDQwNCwgeyJvayI6IEZhbHNlLCAiZXJyb3IiOiAibm90IGZvdW5kIn0pCgogICAgZGVmIF9wcm94eV9wb3N0KHNlbGYpOgogICAgICAgIHRyeToKICAgICAgICAgICAgbiA9IGludChzZWxmLmhlYWRlcnMuZ2V0KCJDb250ZW50LUxlbmd0aCIsICIwIikgb3IgIjAiKQogICAgICAgICAgICByYXcgPSBzZWxmLnJmaWxlLnJlYWQobikKICAgICAgICAgICAgcmVxID0gdXJsbGliLnJlcXVlc3QuUmVxdWVzdCgKICAgICAgICAgICAgICAgIGYie0FQSV9CQVNFfXtzZWxmLnBhdGh9IiwgZGF0YT1yYXcsCiAgICAgICAgICAgICAgICBoZWFkZXJzPXsiQ29udGVudC1UeXBlIjogc2VsZi5oZWFkZXJzLmdldCgiQ29udGVudC1UeXBlIiwgImFwcGxpY2F0aW9uL2pzb24iKX0sCiAgICAgICAgICAgICAgICBtZXRob2Q9IlBPU1QiKQogICAgICAgICAgICB3aXRoIHVybGxpYi5yZXF1ZXN0LnVybG9wZW4ocmVxLCB0aW1lb3V0PTMwMCkgYXMgcjoKICAgICAgICAgICAgICAgIGJvZHkgPSByLnJlYWQoKQogICAgICAgICAgICAgICAgY3R5cGUgPSByLmhlYWRlcnMuZ2V0KCJDb250ZW50LVR5cGUiLCAiYXBwbGljYXRpb24vb2N0ZXQtc3RyZWFtIikKICAgICAgICAgICAgc2VsZi5zZW5kX3Jlc3BvbnNlKDIwMCk7IHNlbGYuX2NvcnMoKQogICAgICAgICAgICBzZWxmLnNlbmRfaGVhZGVyKCJDb250ZW50LVR5cGUiLCBjdHlwZSkKICAgICAgICAgICAgc2VsZi5zZW5kX2hlYWRlcigiQ29udGVudC1MZW5ndGgiLCBzdHIobGVuKGJvZHkpKSkKICAgICAgICAgICAgc2VsZi5lbmRfaGVhZGVycygpCiAgICAgICAgICAgIHNlbGYud2ZpbGUud3JpdGUoYm9keSkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIHNlbGYuX2pzb24oNTAyLCB7Im9rIjogRmFsc2UsICJlcnJvciI6IHN0cihlKX0pCgoKY2xhc3MgVGhyZWFkaW5nU2VydmVyKHNvY2tldHNlcnZlci5UaHJlYWRpbmdNaXhJbiwgaHR0cC5zZXJ2ZXIuSFRUUFNlcnZlcik6CiAgICBkYWVtb25fdGhyZWFkcyA9IFRydWUKCgpkZWYgbWFpbigpOgogICAgc3J2ID0gVGhyZWFkaW5nU2VydmVyKCgiMC4wLjAuMCIsIEJSSURHRV9QT1JUKSwgSGFuZGxlcikKICAgIHByaW50KGYiW2JyaWRnZV0gVm9pY2VXZWF2ZXIg6Zuy56uv5qmL5o6l5ZWf5YuVIDp7QlJJREdFX1BPUlR9IOKGkiBhcGlfdjIge0FQSV9CQVNFfSIsIGZsdXNoPVRydWUpCiAgICBwcmludChmIlticmlkZ2VdIOaooeWei+a4heWWru+8mntISVNUT1JZX1BBVEh9IiwgZmx1c2g9VHJ1ZSkKICAgIHNydi5zZXJ2ZV9mb3JldmVyKCkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=='))
env=dict(os.environ, GSV_ROOT=ROOT, HISTORY_PATH=f'{ROOT}/user_models/model_history.json',
         API_BASE='http://127.0.0.1:9880', IMAGE_BASE='http://127.0.0.1:9881', TEXT_BASE='http://127.0.0.1:9882',
         BRIDGE_PORT='9879')
procs['bridge']=subprocess.Popen(['python','/content/bridge.py'],env=env,
    stdout=open('/content/bridge.log','w'),stderr=subprocess.STDOUT)
time.sleep(3)
with urllib.request.urlopen('http://127.0.0.1:9879/health',timeout=5) as r:
    print('✅ 橋接就緒：', r.read().decode())


## ⑦ 開公開網址（ngrok，授權碼從雲端自動取用）

授權碼**存在你的 VoiceWeaver 帳號雲端**（在 App／網頁的「ngrok 雲端通道」填一次即可），
Colab 只需要一組**非機密的配對碼**就能自動取回授權碼——你不用把授權碼貼進 Colab。

1. App／網頁 → 設定 → **ngrok 雲端通道** → 填 authtoken（＋建議填固定網域）→ 複製「配對碼」。
2. Colab 左側 **🔑 Secrets → 新增** 名稱 `VW_PAIR_CODE`，值貼上那組配對碼（打開 Notebook access）。
   （或不設 Secret，執行本格時會跳出輸入框讓你貼配對碼。）

> 之後每次重開 Colab 都自動從雲端拿授權碼；換 ngrok 授權碼只要在 App／網頁改，Colab 不用動。


In [ ]:
!pip -q install pyngrok
from pyngrok import ngrok, conf
import getpass, json, urllib.request

FIREBASE_PROJECT = 'voiceweaver-af4da'   # 公開的專案 ID

def _secret(name):
    try:
        from google.colab import userdata; return (userdata.get(name) or '').strip()
    except Exception:
        return ''

# 配對碼（非機密）：優先讀 Colab Secret『VW_PAIR_CODE』，否則執行時輸入。
pair = _secret('VW_PAIR_CODE') or input('貼上 App／網頁的「配對碼」：').strip()
assert pair, '需要配對碼'

# 用配對碼從 Firestore 取回你在 App／網頁填的 ngrok 授權碼（該文件只有拿到配對碼才讀得到）。
url = f'https://firestore.googleapis.com/v1/projects/{FIREBASE_PROJECT}/databases/(default)/documents/cloudbridge/{pair}'
with urllib.request.urlopen(url, timeout=15) as r:
    fields = json.load(r).get('fields', {})
token  = fields.get('ngrokToken', {}).get('stringValue', '').strip()
domain = fields.get('ngrokDomain', {}).get('stringValue', '').strip()
assert token, '雲端沒有 ngrok 授權碼——請先在 App／網頁的「ngrok 雲端通道」填入並確認已同步'
print('✅ 已從雲端取回 ngrok 授權碼（未顯示內容）', '｜固定網域：'+domain if domain else '｜無固定網域')

conf.get_default().auth_token = token
ngrok.kill()
opts = {'addr': 9879, 'proto': 'http'}
if domain: opts['domain'] = domain
pub = ngrok.connect(**opts).public_url
print('\n'+'='*54+f'\n  公開網址：{pub}\n'+'='*54)
print('\n📱 App：設定 → GPT-SoVITS API 位址 →「新增」貼上：\n   ', pub)
print('\n🌐 網頁：設定 → 讓電腦幫忙跑運算 →「新增雲端」貼上：\n   ', pub)
print('\n（固定網域下此網址永不變，App／網頁填一次即可）' if domain else '\n（未設固定網域：網址每次會變，建議在 App／網頁填固定網域）')


## 使用說明

**ngrok 授權碼完全不進 Colab**：在 App／網頁的「ngrok 雲端通道」填一次 → 存你的帳號雲端 → Colab 用配對碼自動取回。

| 客戶端 | 填哪裡 |
|---|---|
| 📱 Android App | 設定 → GPT-SoVITS API 位址 → 新增 → 貼 ngrok 網址（可多台、失敗接手）|
| 🌐 網頁版 | 設定 → 讓電腦幫忙跑運算 → 新增雲端 → 貼 ngrok 網址 |
| ☁️ ngrok 授權碼 | App／網頁 → 設定 → ngrok 雲端通道（存雲端，跨裝置同步）|

- **自由選擇功能**：回儲存格 ① 改 `RUN_VOICE/IMAGE/TEXT` 再重跑。
- **自由新增多台**：App 與網頁都可加多個網址，連不上自動換下一個。
- 設了 ngrok **固定網域**，App／網頁填一次網址永不變；否則每次重開要重貼。
- Colab 免費版閒置約 90 分鐘、單次最長約 12 小時回收；回收後重跑 ⑤⑥⑦。
